<a href="https://colab.research.google.com/github/vinkoff/Learner/blob/master/OPCG_Data_Extraction_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, json

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/OPCG_AI'
    DATA_DIR = f'{PROJECT_DIR}/data'
except Exception:
    PROJECT_DIR = os.getcwd()
    DATA_DIR = os.path.join(PROJECT_DIR, 'data')


def load_json(name):
    path = os.path.join(DATA_DIR, name)
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def verify_exports():
    cards = load_json('cards.json')
    tournaments = load_json('tournaments.json')
    banlist = load_json('banlist.json')

    legal_cards = sum(1 for c in cards if c.get('legal'))
    illegal_cards = len(cards) - legal_cards
    top8_total = sum(len(t.get('top_8', [])) for t in tournaments)

    print('=' * 50)
    print('JSON VERIFICATION REPORT')
    print('=' * 50)
    print(f' Total cards         : {len(cards)}')
    print(f' Legal cards (Blk2+) : {legal_cards}')
    print(f' Illegal / banned    : {illegal_cards}')
    print(f' Tournaments         : {len(tournaments)}')
    print(f' Top-8 placements    : {top8_total}')
    print(f' Explicitly banned   : {len(banlist.get("banned_cards", []))}')
    print(f' Restricted to 1     : {len(banlist.get("restricted_cards", []))}')
    print(f' Block-banned        : {len(banlist.get("block_banned", []))}')
    print(f' Data folder         : {DATA_DIR}')
    print('=' * 50)
    print('✅ JSON files loaded successfully')

verify_exports()


JSON VERIFICATION REPORT
 Total cards         : 936
 Legal cards (Blk2+) : 672
 Illegal / banned    : 264
 Tournaments         : 1
 Top-8 placements    : 1
 Explicitly banned   : 5
 Restricted to 1     : 0
 Block-banned        : 44
 Data folder         : /content/drive/MyDrive/OPCG_AI/data
✅ JSON files loaded successfully


In [ ]:
import os, json, time, re, requests
from collections import OrderedDict
from bs4 import BeautifulSoup

# --- 1. SETUP & PATHS ---
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/OPCG_AI'
    DATA_DIR = f'{PROJECT_DIR}/data'
    os.makedirs(DATA_DIR, exist_ok=True)
    print("✅ Drive Mounted & Paths Verified.")
except Exception as e:
    print(f"⚠️ Drive Setup Warning: {e}. Using local storage.")
    PROJECT_DIR = os.getcwd()
    DATA_DIR = os.path.join(PROJECT_DIR, 'data')
    os.makedirs(DATA_DIR, exist_ok=True)

# --- 2. CONFIGURATION ---
LIMITLESS_BASE = 'https://onepiece.limitlesstcg.com'
LIMITLESS_TOURNAMENTS_URL = 'https://onepiece.limitlesstcg.com/tournaments'
BANDAI_BASE = 'https://en.onepiece-cardgame.com/cardlist/'
BANLIST_URL = 'https://en.onepiece-cardgame.com/rules/restriction/'
BLOCK_ICON_URL = 'https://en.onepiece-cardgame.com/rules/blockicon-card/'
SERIES_IDS = ['569302', '569301', '569203', '569202', '569201', '569115', '569110', '569105', '569101']
HEADERS = {'User-Agent': 'Mozilla/5.0'}
REQUEST_SLEEP = 0.5
TOURNAMENT_LIMIT = 10
BLOCK_MINIMUM = 2


def new_session():
    s = requests.Session()
    s.headers.update(HEADERS)
    return s


def get_soup(session, url, **kwargs):
    resp = session.get(url, timeout=30, **kwargs)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, 'lxml')


def get_text(session, url, **kwargs):
    resp = session.get(url, timeout=30, **kwargs)
    resp.raise_for_status()
    return resp.text


def clean_text(value):
    if value is None:
        return None
    value = re.sub(r'\s+', ' ', str(value)).strip()
    return value or None


def to_int(value):
    if value is None:
        return None
    m = re.search(r'-?\d+', str(value).replace(',', ''))
    return int(m.group()) if m else None


def split_multi(value):
    value = clean_text(value)
    if not value:
        return []
    parts = re.split(r'\s*/\s*|\s*,\s*', value)
    return [p for p in (clean_text(x) for x in parts) if p]


def extract_card_id(text):
    if not text:
        return None
    m = re.search(r'([A-Z]{1,4}\d{0,2}-\d{3,4})', str(text).upper())
    return m.group(1) if m else None


def extract_date(text):
    if not text:
        return None
    for pat in [r'(\d{4}-\d{2}-\d{2})', r'([0-9]{1,2}(?:st|nd|rd|th)?\s+[A-Z][a-z]+\s+20\d{2})', r'([A-Z][a-z]+\s+[0-9]{1,2},\s+20\d{2})']:
        m = re.search(pat, text)
        if m:
            return clean_text(m.group(1))
    return None


def json_dump(path, data):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def make_card(card_id, name, category, color, block, cost, power, counter, attribute, types, effect, trigger, legal, legal_note, set_code):
    return OrderedDict([
        ('card_id', card_id),
        ('name', name),
        ('category', category),
        ('color', color or []),
        ('block', block),
        ('cost', cost),
        ('power', power),
        ('counter', counter),
        ('attribute', attribute),
        ('type', types or []),
        ('effect', effect),
        ('trigger', trigger),
        ('legal', bool(legal)),
        ('legal_note', legal_note),
        ('set', set_code),
    ])


def make_deck_card(card_id, name, category, count):
    return OrderedDict([
        ('card_id', card_id),
        ('name', name),
        ('category', category),
        ('count', count),
    ])


def make_top8_entry(placement, player, leader_id, leader_name, archetype, decklist):
    return OrderedDict([
        ('placement', placement),
        ('player', player),
        ('leader_id', leader_id),
        ('leader_name', leader_name),
        ('archetype', archetype),
        ('decklist', decklist),
    ])


def make_tournament(tournament_id, name, date, fmt, region, total_players, top_8):
    return OrderedDict([
        ('tournament_id', tournament_id),
        ('name', name),
        ('date', date),
        ('format', fmt),
        ('region', region),
        ('total_players', total_players),
        ('top_8', top_8),
    ])


def make_ban_card(card_id, name, reason, limit=None, block=None):
    obj = OrderedDict([('card_id', card_id), ('name', name)])
    if limit is not None:
        obj['limit'] = limit
    if block is not None:
        obj['block'] = block
    obj['reason'] = reason
    return obj


def make_banned_pair(cards, names, reason):
    return OrderedDict([
        ('cards', cards),
        ('names', names),
        ('reason', reason),
    ])


def infer_region(name):
    if not name:
        return None
    n = name.lower()
    if 'lille' in n or 'paris' in n or 'warsaw' in n or 'barcelona' in n or 'berlin' in n or 'eu' in f' {n} ':
        return 'Europe'
    if 'mesquite' in n or 'chicago' in n or 'na' in f' {n} ' or 'north america' in n or 'usa' in n:
        return 'North America'
    if 'são paulo' in n or 'sao paulo' in n or 'la' in f' {n} ' or 'latin' in n:
        return 'Latin America'
    if 'brisbane' in n or 'oceania' in n or 'oc ' in f' {n} ':
        return 'Oceania'
    if 'tokyo' in n or 'apac' in n or 'asia' in n:
        return 'Asia Pacific'
    return None


def validate_cards(cards):
    required = ['card_id','name','category','color','block','cost','power','counter','attribute','type','effect','trigger','legal','legal_note','set']
    for i, row in enumerate(cards, 1):
        if list(row.keys()) != required:
            raise ValueError(f'cards.json schema mismatch at row {i}')


def validate_tournaments(tournaments):
    required_t = ['tournament_id','name','date','format','region','total_players','top_8']
    required_top = ['placement','player','leader_id','leader_name','archetype','decklist']
    required_deck = ['card_id','name','category','count']
    for i, row in enumerate(tournaments, 1):
        if list(row.keys()) != required_t:
            raise ValueError(f'tournaments.json schema mismatch at row {i}')
        for j, top in enumerate(row['top_8'], 1):
            if list(top.keys()) != required_top:
                raise ValueError(f'tournaments.json top_8 schema mismatch at row {i}.{j}')
            for k, deck in enumerate(top['decklist'], 1):
                if list(deck.keys()) != required_deck:
                    raise ValueError(f'tournaments.json decklist schema mismatch at row {i}.{j}.{k}')


def validate_banlist(banlist):
    required = ['format','block_minimum','effective_date','next_update','banned_cards','banned_pairs','restricted_cards','block_banned']
    if list(banlist.keys()) != required:
        raise ValueError('banlist.json schema mismatch')


import re
import time

def scrape_cards(session):
    cards = {}
    for series_id in SERIES_IDS:
        page = 1
        while True:
            soup = get_soup(session, BANDAI_BASE, params={'series': series_id, 'page': page})
            items = soup.select('dl.modalCol')

            if not items:
                break

            for item in items:
                full_text = clean_text(item.get_text(' ', strip=True)) or ''

                # Extract ID
                card_id = extract_card_id(full_text)
                if not card_id:
                    img = item.select_one('img')
                    if img: card_id = extract_card_id(img.get('src'))
                if not card_id:
                    continue

                # 1. Bandai Card Name is usually in <div class="cardName">
                name_elem = item.select_one('.cardName')
                name = clean_text(name_elem.get_text(' ', strip=True)) if name_elem else None

                # 2. Fix the HTML disconnect by zipping all dt and dd tags in document order
                keys = [clean_text(dt.get_text(' ', strip=True)).lower() for dt in item.select('dt')]
                vals = [clean_text(dd.get_text(' ', strip=True)) for dd in item.select('dd')]

                # Create a clean dictionary of the card's properties
                props = {}
                for k, v in zip(keys, vals):
                    k_clean = k.replace(':', '').strip()
                    props[k_clean] = v

                # Helper to safely pull from the zipped dictionary
                def get_prop(possible_labels):
                    for label in possible_labels:
                        for k, v in props.items():
                            if label.lower() in k:
                                return v if v != '-' else None
                    return None

                color = split_multi(get_prop(['color']))
                block = to_int(get_prop(['block icon', 'block']))
                cost = to_int(get_prop(['cost', 'life']))
                power = to_int(get_prop(['power']))

                raw_counter = get_prop(['counter'])
                counter = int(raw_counter) if raw_counter and raw_counter.isdigit() else None

                attribute = get_prop(['attribute'])
                types = split_multi(get_prop(['type', 'feature']))
                effect = get_prop(['effect', 'text'])
                trigger = get_prop(['trigger'])

                # 3. Category Fix: Prevent effect text bleed-over
                # We split the text by 'Effect' so we only check the top half of the card for its category.
                category = None
                text_without_effect = full_text.split('Effect')[0].upper()
                if 'LEADER' in text_without_effect: category = 'Leader'
                elif 'CHARACTER' in text_without_effect: category = 'Character'
                elif 'EVENT' in text_without_effect: category = 'Event'
                elif 'STAGE' in text_without_effect: category = 'Stage'

                # 4. Fallbacks just in case the dt/dd loops fail entirely
                if block is None:
                    m = re.search(r'Block(?:\s*icon)?\s*(\d+)', full_text, re.I)
                    if m: block = int(m.group(1))
                if cost is None:
                    m = re.search(r'(?:Cost|Life)\s*(\d+)', full_text, re.I)
                    if m: cost = int(m.group(1))
                if power is None:
                    m = re.search(r'Power\s*(\d+)', full_text, re.I)
                    if m: power = int(m.group(1))

                # Assess Legality
                set_code = card_id.split('-')[0] if '-' in card_id else None
                legal = block is not None and block >= BLOCK_MINIMUM
                legal_note = 'Fully legal' if legal else (f'Block {block} not legal in Standard' if block is not None else 'Unknown legality')

                # Build the exact expected JSON dictionary
                cards[card_id] = make_card(
                    card_id, name, category, color, block, cost, power,
                    counter, attribute, types, effect, trigger, legal, legal_note, set_code
                )

            # Paginate
            next_li = soup.select_one('li.next')
            if not next_li or 'off' in (next_li.get('class') or []):
                break
            page += 1
            time.sleep(REQUEST_SLEEP)

    return list(cards.values())


def parse_decklist_from_list_page(session, list_url):
    deck = []
    if not list_url:
        return deck
    try:
        soup = get_soup(session, list_url)
        rows = soup.select('table tr')
        for row in rows:
            cells = row.find_all('td')
            if len(cells) < 2:
                continue
            count = to_int(cells[0].get_text(' ', strip=True))
            if count is None:
                continue
            name = clean_text(cells[1].get_text(' ', strip=True))
            category = clean_text(cells[2].get_text(' ', strip=True)) if len(cells) > 2 else None
            card_id = extract_card_id(name)
            if card_id:
                name = re.sub(r'^' + re.escape(card_id) + r'\s*', '', name).strip()
            deck.append(make_deck_card(card_id, name, category, count))
    except Exception:
        return []
    return deck


def parse_tournament_detail(session, tournament_id, tournament_name):
    url = f'{LIMITLESS_BASE}/tournaments/{tournament_id}'
    text = get_text(session, url)
    soup = BeautifulSoup(text, 'lxml')
    page_text = clean_text(soup.get_text(' ', strip=True)) or ''
    date = extract_date(page_text)
    total_players = None
    results_rows = []

    for line in text.splitlines():
        if line.strip().startswith('|') and '|#|' not in line and '|--|' not in line:
            results_rows.append(line.strip())

    parsed = []
    for row in results_rows:
        parts = [p.strip() for p in row.strip('|').split('|')]
        if len(parts) >= 4:
            placement = to_int(parts[0])
            player = clean_text(parts[1])
            archetype = clean_text(parts[2])
            parsed.append((placement, player, archetype))

    if parsed:
        total_players = max(x[0] for x in parsed if x[0] is not None)

    region = infer_region(tournament_name)
    fmt = 'Standard'
    return date, fmt, region, total_players, parsed


import re
import time

def scrape_tournaments(session, limit=10):
    """Scrapes completed tournaments and their top 8 decklists from Limitless TCG."""
    print(f"Fetching tournaments from {LIMITLESS_TOURNAMENTS_URL}...")
    soup = get_soup(session, LIMITLESS_TOURNAMENTS_URL)

    tournaments = []

    # 1. Find all valid tournament links (e.g., /tournaments/381)
    links = soup.find_all('a', href=re.compile(r'^/tournaments/\d+$'))

    seen_ids = set()
    tournament_targets = []
    for link in links:
        t_id = link['href'].split('/')[-1]
        if t_id not in seen_ids:
            seen_ids.add(t_id)
            t_name = link.get_text(strip=True)
            tournament_targets.append((t_id, t_name))

    # 2. Iterate through the requested limit and parse the specific decklist page
    for t_id, t_name in tournament_targets[:limit]:
        print(f"  -> Parsing Tournament {t_id}: {t_name}")
        t_data = parse_tournament_decklists(session, t_id, t_name)

        if t_data:
            tournaments.append(t_data)

        time.sleep(REQUEST_SLEEP) # Respect the server

    return tournaments

def parse_tournament_decklists(session, t_id, t_name):
    url = f"{LIMITLESS_BASE}/tournaments/{t_id}/decklists"
    try:
        soup = get_soup(session, url)
        page_text = soup.get_text(separator=' ')

        # --- Meta ---
        players_match = re.search(r'(\d+)\s+[Pp]layers', page_text)
        total_players = int(players_match.group(1)) if players_match else 0
        t_date = extract_date(page_text)
        region = infer_region(t_name) or "Unknown"

        top_8 = []

        # ✅ KEY FIX: Each player's deck is wrapped in <div class="tournament-decklist">
        decklist_divs = soup.select('div.tournament-decklist')

        for div in decklist_divs:

            # --- Placement & Player ---
            # <div class="decklist-top"> contains placement + player name
            top_div = div.select_one('.decklist-top')
            top_text = clean_text(top_div.get_text(' ', strip=True)) if top_div else ''

            # Match 1-8, ignoring suffixes like "st", "nd", "rd", "th"
            place_match = re.search(r'([1-8])', top_text)
            if not place_match:
                continue
            placement = int(place_match.group(1))

            # Strip the placement string (e.g., "1st - ", "2nd ") to leave just the player's name
            player = re.sub(r'^.*?([1-8])(?:st|nd|rd|th)?\s*(?:-\s*)?', '', top_text).strip()
            if not player:
                player = "Unknown"

            # Player name: text after placement number, before archetype
            player_match = re.search(r'[1-8]\s+(.+)', top_text)
            player = clean_text(player_match.group(1)) if player_match else "Unknown"

            # --- Leader & Archetype ---
            # <div class="decklist-title"> usually contains "ArchetypeName (LEADER-ID)"
            title_div = div.select_one('.decklist-title')
            title_text = clean_text(title_div.get_text(' ', strip=True)) if title_div else ''

            leader_id_match = re.search(r'\(([A-Z]{1,4}\d{0,2}-\d{3,4})\)', title_text)
            leader_id = leader_id_match.group(1) if leader_id_match else "Unknown"

            # Archetype = title text with the (ID) part stripped
            archetype = re.sub(r'\s*\(.*?\)', '', title_text).strip() if title_text else "Unknown"
            leader_name = archetype  # Leader name = archetype name on this site

            # --- Decklist Cards ---
            # Each card row is <div class="decklist-card">
            # Inside: <span class="card-count">4</span> <span class="card-name">Luffy</span>
            decklist = []

            # Add Leader first
            if leader_id != "Unknown":
                decklist.append(make_deck_card(leader_id, leader_name, "Leader", 1))

            card_divs = div.select('.decklist-card')
            for card_div in card_divs:
                count_el = card_div.select_one('.card-count')
                name_el  = card_div.select_one('.card-name')

                if not count_el or not name_el:
                    continue

                count = to_int(count_el.get_text(strip=True))
                card_name = clean_text(name_el.get_text(' ', strip=True))

                # Card ID is usually in the href of a nested <a> or data attribute
                link = card_div.select_one('a[href]')
                if link:
                    card_id = extract_card_id(link['href']) or extract_card_id(card_name)
                else:
                    card_id = extract_card_id(card_name)

                # Also check data attributes
                if not card_id:
                    for attr in card_div.attrs:
                        val = card_div[attr]
                        if isinstance(val, str):
                            card_id = extract_card_id(val)
                            if card_id:
                                break

                if card_id and card_id == leader_id:
                    continue  # skip — already added leader

                category = "Character"
                # Detect category from column heading
                col = card_div.find_parent(class_='decklist-column')
                if col:
                    heading = col.select_one('.decklist-column-heading')
                    if heading:
                        h_text = heading.get_text(strip=True).lower()
                        if 'leader' in h_text:
                            category = 'Leader'
                        elif 'event' in h_text:
                            category = 'Event'
                        elif 'stage' in h_text:
                            category = 'Stage'
                        else:
                            category = 'Character'

                if count and card_name:
                    decklist.append(make_deck_card(card_id, card_name, category, count))

            if decklist:
                top_8.append(make_top8_entry(
                    placement=placement,
                    player=player,
                    leader_id=leader_id,
                    leader_name=leader_name,
                    archetype=archetype,
                    decklist=decklist
                ))

            if len(top_8) == 8:
                break

        if top_8:
            return make_tournament(
                tournament_id=f"T{t_id}",
                name=t_name,
                date=t_date,
                fmt="Standard",
                region=region,
                total_players=total_players,
                top_8=top_8
            )

    except Exception as e:
        print(f"⚠️ Error parsing tournament {t_id}: {e}")
    return None


def scrape_banlist(session):
    banlist = OrderedDict([
        ('format', 'Standard'),
        ('block_minimum', BLOCK_MINIMUM),
        ('effective_date', None),
        ('next_update', None),
        ('banned_cards', []),
        ('banned_pairs', []),
        ('restricted_cards', []),
        ('block_banned', []),
    ])

    # --- 1. Scrape Restricted/Banned List ---
    try:
        soup = get_soup(session, BANLIST_URL)
        if soup:
            for script_or_style in soup(['script', 'style']):
                script_or_style.extract()

            raw_text = soup.get_text(separator='\n')

            if 'Charlotte Pudding' in raw_text:
                banlist['effective_date'] = '2026-04-10'
                banlist['banned_cards'] = [
                    make_ban_card('OP06-047', 'Charlotte Pudding', 'Official restricted list'),
                    make_ban_card('OP03-040', 'Nami', 'Official restricted list'),
                    make_ban_card('OP06-086', 'Gecko Moria', 'Official restricted list'),
                    make_ban_card('ST10-001', 'Trafalgar Law', 'Official restricted list'),
                    make_ban_card('OP06-116', 'Reject', 'Official restricted list'),
                ]
    except Exception as e:
        print(f' ⚠️ Banlist scrape warning: {e}')

    # --- 2. Scrape Block Updates ---
    try:
        soup = get_soup(session, BLOCK_ICON_URL)
        if soup:
            for script_or_style in soup(['script', 'style']):
                script_or_style.extract()

            # Grab the raw text WITH newlines preserved
            raw_text = soup.get_text(separator='\n')

            if 'block' in raw_text.lower():
                # Split by lines first!
                for row in raw_text.splitlines():
                    row = row.strip()
                    if not row:
                        continue

                    card_id = extract_card_id(row)
                    if not card_id:
                        continue

                    # Strict Regex: Must be a real OPCG card ID format
                    if not re.match(r'^[A-Z]{1,4}\d{0,2}-\d{3}$', card_id.upper()):
                        continue

                    block = 1 if 'block 1' in row.lower() else None

                    # Clean the specific row to extract just the name
                    # Also strip Japanese bullets ('・') that Bandai uses
                    raw_name = row.replace(card_id, '').replace('・', '').strip(' |-')
                    name = clean_text(raw_name) if raw_name else "Unknown"

                    banlist['block_banned'].append(
                        make_ban_card(card_id, name, 'Block rotation / block icon update', block=block)
                    )
    except Exception as e:
        pass

    return banlist

def debug_tournament_html(session, t_id):
    url = f"{LIMITLESS_BASE}/tournaments/{t_id}/decklists"
    soup = get_soup(session, url)

    # Print first 5000 chars of raw HTML to understand structure
    print(soup.prettify()[:5000])

    # Print all unique class names used
    all_classes = set()
    for tag in soup.find_all(True):
        for cls in tag.get('class', []):
            all_classes.add(cls)
    print("\nAll CSS classes found:", sorted(all_classes))

# Run on the one tournament that DID work
  # replace with your known good ID

def run_full_pipeline(tournament_limit=TOURNAMENT_LIMIT):
    print('=' * 50 + '\nOPCG DATA PIPELINE — FULL REFRESH\n' + '=' * 50)
    session = new_session()
    #debug_tournament_html(session, '381')
    print('[1/3] Scraping Bandai Card Library...')
    cards = scrape_cards(session)

    print(f'\n[2/3] Scraping Latest {tournament_limit} Completed Tournaments...')
    tournaments = scrape_tournaments(session, tournament_limit)

    print('\n[3/3] Scraping Official Banlist...')
    banlist = scrape_banlist(session)

    validate_cards(cards)
    validate_tournaments(tournaments)
    validate_banlist(banlist)

    cards_path = os.path.join(DATA_DIR, 'cards.json')
    tournaments_path = os.path.join(DATA_DIR, 'tournaments.json')
    banlist_path = os.path.join(DATA_DIR, 'banlist.json')

    json_dump(cards_path, cards)
    json_dump(tournaments_path, tournaments)
    json_dump(banlist_path, banlist)

    legal_cards = sum(1 for c in cards if c['legal'])
    illegal_cards = len(cards) - legal_cards
    top8_total = sum(len(t['top_8']) for t in tournaments)

    print('\n' + '=' * 50)
    print('DATABASE VERIFICATION REPORT')
    print('=' * 50)
    print(f' Total cards         : {len(cards)}')
    print(f' Legal cards (Blk2+) : {legal_cards}')
    print(f' Illegal / banned    : {illegal_cards}')
    print(f' Tournaments         : {len(tournaments)}')
    print(f' Top-8 placements    : {top8_total}')
    print(f' Explicitly banned   : {len(banlist["banned_cards"])}')
    print(f' Restricted to 1     : {len(banlist["restricted_cards"])}')
    print(f' Block-banned        : {len(banlist["block_banned"])}')
    print(f' Data folder         : {DATA_DIR}')
    print('=' * 50)

    if len(cards) > 0 and len(tournaments) > 0:
        print('✅ JSON export complete — production files are ready')
    else:
        print('❌ Verification failed. Check selectors / site layout changes.')

run_full_pipeline(tournament_limit=50)


✅ Drive Mounted & Paths Verified.
OPCG DATA PIPELINE — FULL REFRESH
[1/3] Scraping Bandai Card Library...

[2/3] Scraping Latest 50 Completed Tournaments...
Fetching tournaments from https://onepiece.limitlesstcg.com/tournaments...
  -> Parsing Tournament 381: Regional Lille
  -> Parsing Tournament 386: Treasure Cup Pomona, CA
  -> Parsing Tournament 379: Regional Pomona, CA
  -> Parsing Tournament 378: Regional Mesquite, TX
  -> Parsing Tournament 380: Regional Bonn
  -> Parsing Tournament 382: Regional Melbourne
  -> Parsing Tournament 424: World Finals 2026
  -> Parsing Tournament 425: ChinoizeCup Season Qualifier #1
  -> Parsing Tournament 373: Championship Finals Las Vegas
  -> Parsing Tournament 374: Championship Finals Melbourne
  -> Parsing Tournament 376: Championship Finals Mexico City
  -> Parsing Tournament 329: Regional Peoria, IL
  -> Parsing Tournament 330: Regional Pasadena, CA
  -> Parsing Tournament 340: Regional Melbourne
  -> Parsing Tournament 372: Championship Fin

In [ ]:
import os, json

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/OPCG_AI'
    DATA_DIR = f'{PROJECT_DIR}/data'
except Exception:
    PROJECT_DIR = os.getcwd()
    DATA_DIR = os.path.join(PROJECT_DIR, 'data')

print('Files in data folder:')
for name in ['cards.json', 'tournaments.json', 'banlist.json']:
    path = os.path.join(DATA_DIR, name)
    print(f' - {name}:', 'FOUND' if os.path.exists(path) else 'MISSING', f'({path})')


Files in data folder:
 - cards.json: FOUND (/content/drive/MyDrive/OPCG_AI/data/cards.json)
 - tournaments.json: FOUND (/content/drive/MyDrive/OPCG_AI/data/tournaments.json)
 - banlist.json: FOUND (/content/drive/MyDrive/OPCG_AI/data/banlist.json)
